In [8]:
import sys
#sys.path.insert(0, '/lp-dev/24BCE1793')  # on A100
sys.path.insert(0, '..')               # on laptop, uncomment this instead

from src.preprocessing.preprocess import process_file
from src.preprocessing.plv import batch_compute_plv
from src.preprocessing.graph_builder import build_graph_dataset

from src.models.model import FIGNNEncoder, count_parameters
from torch_geometric.loader import DataLoader
import torch

In [4]:
result = process_file(
    "../data/raw/sub-01/ses-S1/eeg/zeroBACK.set",
    label=0
)
print(f"Epochs:   {result['n_epochs']}")
print(f"Channels: {result['n_channels']}")
print(f"Theta shape: {result['band_epochs']['theta'].shape}")

  Loading: zeroBACK.set  label=0


d:\Portfolio Projects\QML EEG Paper\stage-1 implementation\notebooks\..\src\preprocessing\preprocess.py:37: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
d:\Portfolio Projects\QML EEG Paper\stage-1 implementation\notebooks\..\src\preprocessing\preprocess.py:37: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
d:\Portfolio Projects\QML EEG Paper\stage-1 implementation\notebooks\..\src\preprocessing\preprocess.py:37: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(set_path, prel

    → 62 channels  |  198 epochs  |  500.0 Hz
Epochs:   198
Channels: 62
Theta shape: (198, 62, 1000)


In [5]:
# Slice just 3 epochs to test PLV fast
test_bands = {band: result['band_epochs'][band][:3] 
              for band in result['band_epochs']}
test_result = {**result, 'band_epochs': test_bands, 'n_epochs': 3}

plv_list = batch_compute_plv(test_result['band_epochs'])
print(f"PLV list length: {len(plv_list)}")
print(f"PLV matrix shape (delta): {plv_list[0]['delta'].shape}")

    PLV: epoch 0/3
PLV list length: 3
PLV matrix shape (delta): (62, 62)


In [6]:
graphs = build_graph_dataset(test_result, plv_list)
g = graphs[0]
print(f"Nodes: {g.x.shape}")
print(f"Edges: {g.edge_index.shape}")
print(f"Edge features: {g.edge_attr.shape}")
print(f"Label: {g.y}")

    Built 3 graphs  |  sample edges: 2422  |  label: 0
Nodes: torch.Size([62, 4])
Edges: torch.Size([2, 2422])
Edge features: torch.Size([2422, 4])
Label: tensor([0])


In [9]:
model = FIGNNEncoder()
count_parameters(model)

# Make a small batch from your 3 test graphs
loader = DataLoader(graphs, batch_size=3)
batch  = next(iter(loader))

# Forward pass — no training, just checking shapes
model.eval()
with torch.no_grad():
    z = model(batch)

print(f"Output z shape: {z.shape}")
print(f"Expected: torch.Size([3, 32])")

Trainable parameters: 12,100
Output z shape: torch.Size([3, 32])
Expected: torch.Size([3, 32])


In [ ]:
import mne
mne.set_log_level("WARNING")
raw_tmp = mne.io.read_raw_eeglab(
    "../data/raw/sub-01/ses-S1/eeg/zeroBACK.set", 
    preload=False, verbose=False
)
ecg_chs = [ch for ch in raw_tmp.ch_names if "ECG" in ch.upper()]
raw_tmp.drop_channels(ecg_chs)
ch_names = raw_tmp.ch_names  # exactly 62 names

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx
from torch_geometric.utils import to_networkx
import numpy as np

g = graphs[0]

# Convert to networkx
G = to_networkx(g, to_undirected=True)

# Node positions — use actual electrode layout (approximate)
# For now just use spring layout
pos = nx.spring_layout(G, seed=42)

# Node colours = theta band power (index 1)
node_colors = g.x[:, 1].numpy()  # theta power per electrode

# Edge weights = mean PLV across 4 bands
edge_weights = g.edge_attr.mean(dim=1).numpy()

fig, ax = plt.subplots(figsize=(10, 8))
ax.set_facecolor("#0d0d1a")
fig.patch.set_facecolor("#0d0d1a")

# Draw edges
nx.draw_networkx_edges(G, pos, ax=ax,
    edge_color=edge_weights[:len(G.edges())],
    edge_cmap=plt.cm.plasma,
    alpha=0.3, width=0.5)

# Draw nodes
nc = nx.draw_networkx_nodes(G, pos, ax=ax,
    node_color=node_colors,
    cmap=plt.cm.RdYlGn,
    node_size=120)

# Labels
nx.draw_networkx_labels(G, pos, ax=ax,
    labels={i: ch_names[i] for i in range(len(ch_names))},
    font_size=5, font_color="white")

plt.colorbar(nc, ax=ax, label="Theta power", shrink=0.6)
ax.set_title("EEG Functional Connectivity Graph — zeroBACK epoch 0", 
             color="white", fontsize=11)
ax.axis("off")

plt.tight_layout()
plt.savefig("outputs/plots/graph_viz.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved to outputs/plots/graph_viz.png")

KeyError: 62